# Can we predict next-month arrears?
Target: `arrears_next_month`. Follows `checklist.md`.

Key decisions:
- `final_notice_sent` is recorded in the billing cycle *after* the billing month, so it is not known at prediction time. It is also nearly a proxy for the target. **Dropped.**
- `meter_ref` is an identifier: **dropped** as a feature (used only for grouping).
- The model will be used on later months, and accounts repeat across rows, so the test set is the latest months **and** whole held-out accounts.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from sklearn.base import clone
df=pd.read_csv('data.csv',parse_dates=['billing_month'])
print(df.shape, df.arrears_next_month.mean())
df.groupby('final_notice_sent').arrears_next_month.agg(['mean','size'])

## 1. Test set (set aside before any modelling)
Test = rows in the latest 6 billing months (from 2025-07) belonging to a random 20% of accounts held out entirely. Training = remaining accounts, only months before 2025-07 (so nothing in training postdates the test period, and no training account appears in test).

In [ ]:
rng=np.random.RandomState(0)
meters=df.meter_ref.unique()
test_meters=set(rng.choice(meters,int(0.2*len(meters)),replace=False))
CUT=pd.Timestamp('2025-07-01')
is_test=df.meter_ref.isin(test_meters)&(df.billing_month>=CUT)
is_train=~df.meter_ref.isin(test_meters)&(df.billing_month<CUT)
train=df[is_train].sort_values('billing_month').reset_index(drop=True)
test=df[is_test].reset_index(drop=True)
assert not set(train.meter_ref)&set(test.meter_ref)
print(len(train),len(test),train.arrears_next_month.mean(),test.arrears_next_month.mean())

## 2. Features (known at prediction time only)
`tariff`, `occupants`, `kwh`, `bill_eur`, `late_payments_12m`: all describe the billing month or earlier. Dropped: `meter_ref`, `final_notice_sent` (post-dates the billing month), `billing_month` (a calendar position cannot extrapolate to later months).

In [ ]:
FEATS=['tariff','occupants','kwh','bill_eur','late_payments_12m']
CAT=['tariff']; NUM=[f for f in FEATS if f not in CAT]
y=train.arrears_next_month.values; X=train[FEATS]
def prep(): # fitted inside each fold because it is part of the pipeline
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),('n',StandardScaler(),NUM)])

## 3. Candidate models (named before any scoring)
1. `baseline`: predicts the training prevalence for every row (trivial).
2. `logit_late`: logistic regression on `late_payments_12m` alone.
3. `logit_all`: logistic regression, all legitimate features.
4. `gbm_*`: histogram gradient boosting, all legitimate features, small fixed grid over depth/learning rate (chosen by CV).

Metric: average precision (PR-AUC), since arrears is the minority class (~12%). Threshold is chosen on CV out-of-fold predictions to maximise F1.

In [ ]:
def build(k):
    if k=='baseline': return Pipeline([('m',DummyClassifier(strategy='prior'))])
    if k=='logit_late': return Pipeline([('p',ColumnTransformer([('n',StandardScaler(),['late_payments_12m'])])),('m',LogisticRegression(max_iter=1000))])
    if k=='logit_all': return Pipeline([('p',prep()),('m',LogisticRegression(max_iter=1000))])
    return Pipeline([('p',prep()),('m',HistGradientBoostingClassifier(random_state=0,max_iter=150,**CANDS[k]))])
CANDS={'baseline':{},'logit_late':{},'logit_all':{}}
for d in (2,3,None):
    for lr in (0.03,0.1): CANDS[f'gbm_d{d}_lr{lr}']=dict(max_depth=d,learning_rate=lr)
print(list(CANDS))

## 4. Cross-validation inside training rows
Forward-chaining over months: 5 folds, each validating on a later block of months. Training accounts are split into 5 random groups; fold k validates on group k's rows in block k, and trains only on earlier months of the *other* groups. So every fold has earlier-trains/later-tests and whole accounts on each side.

In [ ]:
months=np.sort(train.billing_month.unique())
blocks=np.array_split(months[len(months)//3:],5)   # first third only ever used as training
tm=train.meter_ref.unique(); grp=dict(zip(tm,np.random.RandomState(1).permutation(len(tm))%5))
g=train.meter_ref.map(grp).values
def folds():
    for k,b in enumerate(blocks):
        va=train.billing_month.isin(b).values&(g==k)
        tr=(train.billing_month<b[0]).values&(g!=k)
        yield np.where(tr)[0],np.where(va)[0]
FOLDS=list(folds())
assert all(not set(train.meter_ref.iloc[a])&set(train.meter_ref.iloc[b]) for a,b in FOLDS)
for tr,va in FOLDS: print(len(tr),len(va),y[tr].mean().round(3),y[va].mean().round(3))

In [ ]:
oof={}; rows=[]
for k in CANDS:
    p=np.full(len(train),np.nan); aps=[]
    for tr,va in FOLDS:
        m=clone(build(k)).fit(X.iloc[tr],y[tr]); p[va]=m.predict_proba(X.iloc[va])[:,1]
        aps.append(average_precision_score(y[va],p[va]))
    oof[k]=p; rows.append((k,np.mean(aps),np.std(aps)))
cv=pd.DataFrame(rows,columns=['model','cv_AP','sd']).sort_values('cv_AP',ascending=False); cv

## 5. Choice on CV only (threshold included)

In [ ]:
best=cv.iloc[0].model
mask=~np.isnan(oof[best])
ths=np.linspace(0.05,0.9,86)
f1s=[f1_score(y[mask],oof[best][mask]>=t) for t in ths]
THR=float(ths[int(np.argmax(f1s))])
print(best,'threshold',THR,'CV F1',max(f1s))

**Recorded choice (made from the CV table above, before the test set is touched):** the model with highest mean CV average precision is `best` above; its decision threshold is `THR`, picked to maximise F1 on out-of-fold predictions. Features: `tariff, occupants, kwh, bill_eur, late_payments_12m`. No further changes after seeing test results.

## 6. Refit on all training rows, score test once

In [ ]:
final=clone(build(best)).fit(X,y)
pt=final.predict_proba(test[FEATS])[:,1]; yt=test.arrears_next_month.values
base=clone(build('baseline')).fit(X,y).predict_proba(test[FEATS])[:,1]
print('TEST  model:   AP=%.3f AUC=%.3f F1@thr=%.3f'%(average_precision_score(yt,pt),roc_auc_score(yt,pt),f1_score(yt,pt>=THR)))
print('TEST  baseline: AP=%.3f (= test prevalence %.3f)'%(average_precision_score(yt,base),yt.mean()))

## 7. Conclusion
See the test numbers above (the headline). On the run I did (not saved as outputs, see note): `logit_all` won CV (AP about 0.20 vs 0.065 baseline, high fold variance), threshold 0.05 (edge of the grid), and test AP 0.477 vs 0.217 prevalence, AUC 0.73, F1 0.44. Arrears prevalence climbs from about 2% to 22% over time, so the base rate drifts and CV (earlier, low-prevalence months) understates test AP and badly mis-calibrates the threshold. Limits: the test set covers ~20% of accounts over 6 months; lift is relative to the prevalence baseline. `final_notice_sent` would give a far higher score but is not available when the prediction is needed, so using it would be leakage.